In [ ]:
!pip install torch torchvision matplotlib seaborn scikit-learn

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import time
from sklearn.metrics import confusion_matrix, classification_report
from torch.utils.data import DataLoader

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
CLASSES = ('airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

# Transform: resize ke 224x224 karena VGG butuh ukuran ini
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])  # ImageNet stats
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Download CIFAR-10
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=True, transform=transform_train)
testset  = torchvision.datasets.CIFAR10(root='./data', train=False,
                                         download=True, transform=transform_test)

trainloader = DataLoader(trainset, batch_size=32, shuffle=True,  num_workers=2)
testloader  = DataLoader(testset,  batch_size=32, shuffle=False, num_workers=2)

In [ ]:
print(f"Train samples : {len(trainset)}")
print(f"Test  samples : {len(testset)}")
print(f"Classes       : {CLASSES}")

Train samples : 50000
Test  samples : 10000
Classes       : ('airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')


In [ ]:
def build_vgg_maxpool(num_classes=10):
    """VGG16 pretrained dengan MaxPooling (default bawaan VGG)"""
    model = models.vgg16(pretrained=True)

    # Freeze semua layer kecuali classifier
    for param in model.features.parameters():
        param.requires_grad = False

    # Ganti classifier akhir sesuai jumlah kelas CIFAR-10
    model.classifier[6] = nn.Linear(4096, num_classes)
    return model.to(device)

In [ ]:
def build_vgg_avgpool(num_classes=10):
    """VGG16 pretrained, MaxPool diganti AveragePool di adaptive pooling"""
    model = models.vgg16(pretrained=True)

    # Freeze feature layers
    for param in model.features.parameters():
        param.requires_grad = False

    # Ganti adaptive avgpool (layer antara features dan classifier)
    model.avgpool = nn.AdaptiveAvgPool2d((7, 7))  # sama ukuran output, tapi avg

    # Ganti MaxPool2d di dalam features dengan AvgPool2d
    new_features = []
    for layer in model.features:
        if isinstance(layer, nn.MaxPool2d):
            new_features.append(nn.AvgPool2d(kernel_size=2, stride=2))
        else:
            new_features.append(layer)
    model.features = nn.Sequential(*new_features)

    # Ganti classifier akhir
    model.classifier[6] = nn.Linear(4096, num_classes)
    return model.to(device)

In [ ]:
model_maxpool = build_vgg_maxpool()
model_avgpool = build_vgg_avgpool()

print("✅ Model VGG16-MaxPool siap")
print("✅ Model VGG16-AvgPool siap")
print(f"\nTotal parameter MaxPool: {sum(p.numel() for p in model_maxpool.parameters()):,}")
print(f"Total parameter AvgPool: {sum(p.numel() for p in model_avgpool.parameters()):,}")

✅ Model VGG16-MaxPool siap
✅ Model VGG16-AvgPool siap

Total parameter MaxPool: 134,301,514
Total parameter AvgPool: 134,301,514


In [ ]:
def train_model(model, trainloader, testloader, epochs=5, lr=0.001, model_name="Model"):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [],   'val_acc': [],
        'epoch_time': []
    }

    for epoch in range(epochs):
        start_time = time.time()
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        train_loss = running_loss / len(trainloader)
        train_acc  = 100. * correct / total

        # Validasi
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in testloader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss    = criterion(outputs, labels)
                val_loss    += loss.item()
                _, predicted = outputs.max(1)
                val_total   += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        val_loss = val_loss / len(testloader)
        val_acc  = 100. * val_correct / val_total
        elapsed  = time.time() - start_time

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['epoch_time'].append(elapsed)

        print(f"[{model_name}] Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.2f}% | "
              f"Time: {elapsed:.1f}s")

    return history

In [ ]:
def get_predictions(model, testloader):
    """Ambil semua prediksi dan label asli"""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in testloader:
            images = images.to(device)
            outputs = model(images)
            _, preds = outputs.max(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds)

In [ ]:
EPOCHS = 5  # Naikkan ke 10-15 untuk hasil lebih baik

print("=" * 60)
print("TRAINING VGG16 dengan MaxPooling")
print("=" * 60)
history_max = train_model(model_maxpool, trainloader, testloader,
                           epochs=EPOCHS, model_name="VGG-MaxPool")

print("\n" + "=" * 60)
print("TRAINING VGG16 dengan AveragePooling")
print("=" * 60)
history_avg = train_model(model_avgpool, trainloader, testloader,
                           epochs=EPOCHS, model_name="VGG-AvgPool")

print("\n✅ Training selesai!")

TRAINING VGG16 dengan MaxPooling


In [ ]:

epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Accuracy ---
axes[0].plot(epochs_range, history_max['val_acc'], 'b-o', label='VGG-MaxPool')
axes[0].plot(epochs_range, history_avg['val_acc'], 'r-s', label='VGG-AvgPool')
axes[0].set_title('Validation Accuracy Comparison', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy (%)')
axes[0].legend()
axes[0].grid(True)

# --- Loss ---
axes[1].plot(epochs_range, history_max['val_loss'], 'b-o', label='VGG-MaxPool')
axes[1].plot(epochs_range, history_avg['val_loss'], 'r-s', label='VGG-AvgPool')
axes[1].set_title('Validation Loss Comparison', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('accuracy_loss_comparison.png', dpi=150)
plt.show()

In [ ]:

labels_true_max, preds_max = get_predictions(model_maxpool, testloader)
labels_true_avg, preds_avg = get_predictions(model_avgpool, testloader)

cm_max = confusion_matrix(labels_true_max, preds_max)
cm_avg = confusion_matrix(labels_true_avg, preds_avg)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, cm, title in zip(axes,
                          [cm_max, cm_avg],
                          ['VGG16 - MaxPooling', 'VGG16 - AveragePooling']):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print("📌 Confusion matrix disimpan: confusion_matrix.png")

In [ ]:
def final_accuracy(history):
    return history['val_acc'][-1]

def total_time(history):
    return sum(history['epoch_time'])

avg_time_max = np.mean(history_max['epoch_time'])
avg_time_avg = np.mean(history_avg['epoch_time'])

acc_max = final_accuracy(history_max)
acc_avg = final_accuracy(history_avg)

# Classification Report
print("=" * 60)
print("CLASSIFICATION REPORT — VGG16 MaxPooling")
print("=" * 60)
print(classification_report(labels_true_max, preds_max, target_names=CLASSES))

print("=" * 60)
print("CLASSIFICATION REPORT — VGG16 AveragePooling")
print("=" * 60)
print(classification_report(labels_true_avg, preds_avg, target_names=CLASSES))

# Bar chart perbandingan
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

models_name = ['VGG-MaxPool', 'VGG-AvgPool']
colors = ['steelblue', 'tomato']

# Accuracy bar
axes[0].bar(models_name, [acc_max, acc_avg], color=colors, width=0.4)
axes[0].set_title('Final Validation Accuracy', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_ylim(0, 100)
for i, v in enumerate([acc_max, acc_avg]):
    axes[0].text(i, v + 1, f"{v:.2f}%", ha='center', fontweight='bold')

# Time bar
axes[1].bar(models_name, [avg_time_max, avg_time_avg], color=colors, width=0.4)
axes[1].set_title('Average Training Time per Epoch (s)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Time (seconds)')
for i, v in enumerate([avg_time_max, avg_time_avg]):
    axes[1].text(i, v + 0.5, f"{v:.1f}s", ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('summary_comparison.png', dpi=150)
plt.show()

print("\n" + "=" * 60)
print("RINGKASAN PERBANDINGAN AKHIR")
print("=" * 60)
print(f"{'Metric':<30} {'VGG-MaxPool':>15} {'VGG-AvgPool':>15}")
print("-" * 60)
print(f"{'Final Val Accuracy (%)':<30} {acc_max:>15.2f} {acc_avg:>15.2f}")
print(f"{'Avg Time/Epoch (s)':<30} {avg_time_max:>15.2f} {avg_time_avg:>15.2f}")
print(f"{'Total Training Time (s)':<30} {total_time(history_max):>15.1f} {total_time(history_avg):>15.1f}")
print("-" * 60)
winner_acc  = "MaxPool" if acc_max > acc_avg else "AvgPool"
winner_time = "MaxPool" if avg_time_max < avg_time_avg else "AvgPool"
print(f"🏆 Akurasi lebih tinggi : VGG-{winner_acc}")
print(f"⚡ Lebih cepat          : VGG-{winner_time}")